# 03 · Model Training & Prediction

**Goal:** train Linear Regression, Random Forest, and XGBoost regressors to predict
smartphone price, evaluate each, compare results, and use the best model to derive
business insights (price sensitivity, brand premium).

This notebook loads the splits/encoders/scaler saved by `02_preprocessing.ipynb` from
`artifacts/` — run that notebook first if you haven't already.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ARTIFACTS_DIR = Path("artifacts")

## 3.0 Load preprocessed data

In [ ]:
X_train      = joblib.load(ARTIFACTS_DIR / "X_train.pkl")
X_test       = joblib.load(ARTIFACTS_DIR / "X_test.pkl")
y_train      = joblib.load(ARTIFACTS_DIR / "y_train.pkl")
y_test       = joblib.load(ARTIFACTS_DIR / "y_test.pkl")
X_train_lr   = joblib.load(ARTIFACTS_DIR / "X_train_lr.pkl")
X_test_lr    = joblib.load(ARTIFACTS_DIR / "X_test_lr.pkl")
X_train_tree = joblib.load(ARTIFACTS_DIR / "X_train_tree.pkl")
X_test_tree  = joblib.load(ARTIFACTS_DIR / "X_test_tree.pkl")
scaler       = joblib.load(ARTIFACTS_DIR / "scaler.pkl")
encoders     = joblib.load(ARTIFACTS_DIR / "encoders.pkl")

print("Training shape:", X_train.shape)
print("Testing shape :", X_test.shape)

## 3.1 Evaluation helper

In [ ]:
def evaluate_model(name, y_true, y_pred):
    """Print MAE / RMSE / R2 and plot Actual-vs-Predicted + Residuals."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)

    print(f"\n{name}")
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R2  :", r2)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    # Actual vs Predicted
    axes[0].scatter(y_true, y_pred, alpha=0.6)
    axes[0].plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], "r--")
    axes[0].set_xlabel("Actual Price")
    axes[0].set_ylabel("Predicted Price")
    axes[0].set_title(f"{name}: Actual vs Predicted")

    # Residual plot
    residuals = y_true - y_pred
    axes[1].scatter(y_pred, residuals, alpha=0.6)
    axes[1].axhline(y=0, color="red", linestyle="--")
    axes[1].set_xlabel("Predicted Price")
    axes[1].set_ylabel("Residuals")
    axes[1].set_title(f"{name}: Residual Plot")

    plt.tight_layout()
    plt.show()

    return mae, rmse, r2

## 3.2 Linear Regression (uses scaled features)

In [ ]:
lin_model = LinearRegression()
lin_model.fit(X_train_lr, y_train)
y_pred_lr = lin_model.predict(X_test_lr)
mae_lr, rmse_lr, r2_lr = evaluate_model("Linear Regression", y_test, y_pred_lr)

## 3.3 Random Forest Regressor (uses raw features)

In [ ]:
rf_model = RandomForestRegressor(
    random_state=42, n_estimators=300, max_depth=5, min_samples_leaf=1, min_samples_split=2
)
rf_model.fit(X_train_tree, y_train)
y_pred_rf = rf_model.predict(X_test_tree)
mae_rf, rmse_rf, r2_rf = evaluate_model("Random Forest", y_test, y_pred_rf)

## 3.4 XGBoost Regressor (uses raw features)

In [ ]:
xgb_model = XGBRegressor(
    random_state=42,
    n_estimators=500,
    learning_rate=0.01,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
)
xgb_model.fit(X_train_tree, y_train)
y_pred_xgb = xgb_model.predict(X_test_tree)
mae_xgb, rmse_xgb, r2_xgb = evaluate_model("XGBoost", y_test, y_pred_xgb)

## 3.5 Model comparison table

In [ ]:
comparison = pd.DataFrame({
    "Model":    ["Linear Regression", "Random Forest", "XGBoost"],
    "MAE":      [mae_lr, mae_rf, mae_xgb],
    "RMSE":     [rmse_lr, rmse_rf, rmse_xgb],
    "R2 Score": [r2_lr, r2_rf, r2_xgb],
}).sort_values("R2 Score", ascending=False).reset_index(drop=True)

print("Model Comparison:")
comparison


import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

bars = plt.bar(comparison["Model"], comparison["R2 Score"])

plt.title("Model Comparison - R² Score")
plt.xlabel("Model")
plt.ylabel("R² Score")

plt.xticks(rotation=15)

# Add values on top of each bar
for bar in bars:
    value = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.01,
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )

plt.ylim(0, max(comparison["R2 Score"]) + 0.08)

plt.tight_layout()
plt.show()


import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

x = range(len(comparison))
width = 0.35

# MAE bars
bars_mae = plt.bar(
    [i - width/2 for i in x],
    comparison["MAE"],
    width=width,
    label="MAE"
)

# RMSE bars
bars_rmse = plt.bar(
    [i + width/2 for i in x],
    comparison["RMSE"],
    width=width,
    label="RMSE"
)

plt.title("Model Comparison - MAE vs RMSE")
plt.xlabel("Model")
plt.ylabel("Error")
plt.xticks(x, comparison["Model"], rotation=15)
plt.legend()

# Add MAE values on top
for bar in bars_mae:
    value = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.01,
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )

# Add RMSE values on top
for bar in bars_rmse:
    value = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.01,
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )

plt.tight_layout()
plt.show()

## 3.6 Feature importance

Which inputs matter most to each model? Random Forest and XGBoost expose
`feature_importances_` directly (based on how much each feature reduces error across the
trees). Linear Regression doesn't have that concept, but since its inputs were standardized
(`X_train_lr`), the absolute value of each coefficient is a fair stand-in for importance —
larger magnitude means a bigger swing in predicted price per unit of that (scaled)
feature.

In [ ]:
feature_names = X_train.columns

rf_importances  = pd.Series(rf_model.feature_importances_, index=feature_names).sort_values(ascending=False)
xgb_importances = pd.Series(xgb_model.feature_importances_, index=feature_names).sort_values(ascending=False)
lin_importances = pd.Series(np.abs(lin_model.coef_), index=feature_names).sort_values(ascending=False)

print("Random Forest — feature importances:")
print(rf_importances)

print("\nXGBoost — feature importances:")
print(xgb_importances)

print("\nLinear Regression — |standardized coefficient|:")
print(lin_importances)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

rf_importances.plot(kind="barh", ax=axes[0], color="forestgreen", edgecolor="black")
axes[0].invert_yaxis()
axes[0].set_title("Random Forest")
axes[0].set_xlabel("Importance")

xgb_importances.plot(kind="barh", ax=axes[1], color="royalblue", edgecolor="black")
axes[1].invert_yaxis()
axes[1].set_title("XGBoost")
axes[1].set_xlabel("Importance")

lin_importances.plot(kind="barh", ax=axes[2], color="indianred", edgecolor="black")
axes[2].invert_yaxis()
axes[2].set_title("Linear Regression")
axes[2].set_xlabel("|Standardized Coefficient|")

fig.suptitle("Feature Importance by Model", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

## 3.7 Save trained models

Saved so the models can be reused for inference elsewhere without retraining.

In [ ]:
joblib.dump(lin_model, ARTIFACTS_DIR / "lin_model.pkl")
joblib.dump(rf_model, ARTIFACTS_DIR / "rf_model.pkl")
joblib.dump(xgb_model, ARTIFACTS_DIR / "xgb_model.pkl")
comparison.to_csv(ARTIFACTS_DIR / "model_comparison.csv", index=False)

print("Saved trained models and comparison table to:", ARTIFACTS_DIR.resolve())

---
## Prediction & Insights

The sections below use the trained models to answer two business questions:
1. How much is one extra GB of RAM / storage worth to the predicted price?
2. Which brands command a price premium, according to the model?

### 4.1 Price sensitivity analysis

How much does the model think one extra GB of RAM / storage is worth?
Approach: bump the feature up by 1 for every row in the test set, re-predict, and average
the change in predicted price. Engineered features (`ram_storage`, `total_memory`, etc.)
are recomputed so the comparison stays consistent with how the model was trained.

In [ ]:
def recompute_features(X):
    """Refresh engineered columns after changing ram/storage/battery/rating."""
    X = X.copy()
    X["total_memory"]    = X["ram"] + X["storage"]
    X["ram_storage"]     = X["ram"] * X["storage"]
    X["battery_per_ram"] = X["battery"] / X["ram"]
    X["storage_per_ram"] = X["storage"] / X["ram"]
    X["rating_ram"]      = X["rating"] * X["ram"]
    X["rating_storage"]  = X["rating"] * X["storage"]
    X["rating_battery"]  = X["rating"] * X["battery"]
    return X

def marginal_effect(model, X, feature, step=1, scaler=None):
    """Average predicted-price change from a +step bump in `feature`."""
    X_base = recompute_features(X)
    X_high = X.copy()
    X_high[feature] = X_high[feature] + step
    X_high = recompute_features(X_high)

    if scaler is not None:          # Linear Regression needs scaled input
        pred_base = model.predict(scaler.transform(X_base))
        pred_high = model.predict(scaler.transform(X_high))
    else:                           # tree models use raw features
        pred_base = model.predict(X_base)
        pred_high = model.predict(X_high)

    return (pred_high - pred_base).mean()

sensitivity = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest", "XGBoost"],
    "Price per +1GB RAM (Rs)": [
        marginal_effect(lin_model, X_test, "ram", scaler=scaler),
        marginal_effect(rf_model, X_test_tree, "ram"),
        marginal_effect(xgb_model, X_test_tree, "ram"),
    ],
    "Price per +1GB Storage (Rs)": [
        marginal_effect(lin_model, X_test, "storage", scaler=scaler),
        marginal_effect(rf_model, X_test_tree, "storage"),
        marginal_effect(xgb_model, X_test_tree, "storage"),
    ],
})

print(sensitivity)

sensitivity.set_index("Model").plot(kind="bar", figsize=(8, 5), edgecolor="black")
plt.title("Price Sensitivity: Effect of +1GB RAM / Storage on Predicted Price")
plt.ylabel("Change in Predicted Price (Rs)")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### 4.2 Brand premium quantification

Average predicted price by brand, using the best-performing model (whichever row is on top
of the `comparison` table above).

In [ ]:
best_model_name = comparison.iloc[0]["Model"]
best_model = {"Linear Regression": lin_model, "Random Forest": rf_model, "XGBoost": xgb_model}[best_model_name]

# Predict on the test set with the right feature format for that model
if best_model_name == "Linear Regression":
    y_pred_best = best_model.predict(X_test_lr)
else:
    y_pred_best = best_model.predict(X_test_tree)

# Decode the brand column back from numbers to names
brand_names = encoders["brand"].inverse_transform(X_test["brand"])

brand_premium = (
    pd.DataFrame({"brand": brand_names, "predicted_price": y_pred_best})
    .groupby("brand")["predicted_price"]
    .mean()
    .sort_values(ascending=False)
)

print(brand_premium)

plt.figure(figsize=(9, 5))
brand_premium.plot(kind="bar", edgecolor="black", color="mediumseagreen")
plt.title(f"Brand Premium: Average Predicted Price by Brand ({best_model_name})")
plt.ylabel("Average Predicted Price (Rs)")
plt.xlabel("Brand")
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()